# 01_fundamentals: Call It, Prompt It, Structure It, Loop It

[Open in Colab](https://colab.research.google.com/github/Utkarsh-09/AI_GURU_labs/blob/main/notebooks/01_fundamentals.ipynb)

**Session:** Day 1, S1 — Fundamentals block (90-minute block, compressible to 45)
**Expected runtime:** **60 minutes** of participant time on the FULL path, **45 minutes** on the COMPRESSED path (both include the 10-minute diagnostic). Machine time is about **3 minutes** on a Colab free-tier CPU runtime or a laptop: every model call is a short hosted-API request. No GPU anywhere in this notebook.
**Needs:** `OPENAI_API_KEY` — in `.env` at the repo root locally, in **Colab Secrets** (key icon, left sidebar) on Colab. Data files come with the repo: `data/eval/heldout_20.jsonl`, `data/finetune/ticket_schema.json`, `corpus/tickets/tickets_raw.jsonl`.
**A correct result looks like:** the last cell prints `01 FUNDAMENTALS OK` with your diagnostic verdict, `5 / 5` schema-valid ticket records, and an agent that reached a final answer in at most 4 steps.

> All data in this lab is synthetic. No real OQ material anywhere.

---

## TWO PATHS IN ONE NOTEBOOK — read this before running anything

The first 10 minutes are a **diagnostic**. Its result decides, for the whole room, which path S1 takes. The facilitator announces the path. You do not pick it yourself.

| Part | Where | FULL path (60 min) | COMPRESSED path (45 min) |
|---|---|---|---|
| **A. Setup + diagnostic** | top of the notebook to **THE FORK** | run — 10 min | run — 10 min |
| **B. Fundamentals**: the raw API call, roles, memory, tokens, temperature, prompting | **THE FORK** to **PART C STARTS HERE**; every cell there is marked `[FULL PATH ONLY]` | run — 20 min | **SKIP** |
| **C. Structured outputs + the agent loop** | **PART C STARTS HERE** to the end | run — 30 min | run — 35 min |

**COMPRESSED path, in one line:** after the diagnostic, scroll to the cell headed **PART C STARTS HERE**, click the code cell under it, and use *Runtime → Run after* (Colab) or *Run → Run Selected Cell and All Below* (Jupyter). Nothing in Part C uses anything from Part B.

If you see `# [FULL PATH ONLY]` at the top of a cell on the compressed path, you are in the wrong place: scroll down to Part C.

If the room compresses, the reclaimed 45 minutes runs the Claude Code configuration session (`facilitator/claude_code_session.md`).

**Why this cell:** one notebook has to run in two places — Colab during
the program, and a local machine as the fallback. This first code cell
detects which one it woke up in, mounts Google Drive on Colab so
checkpoints survive a disconnect, and sets the three variables every
later cell can rely on: `IN_COLAB`, `REPO_ROOT`, `CHECKPOINT_DIR`.

In [ ]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

**Why this cell:** each notebook installs only what it needs, with exact
pins matching `requirements.txt`. Colab preinstalls all three of these
at these versions, so on Colab this is a no-op that takes a few seconds;
locally you installed `requirements.txt` during setup.

In [ ]:
# Pinned installs — versions match requirements.txt. Colab only.
if IN_COLAB:
    %pip install -q requests==2.32.4 python-dotenv==1.2.3 jsonschema==4.26.0
print("Install cell done.")

**Why this cell:** the hosted model needs an API key, and the key must
never be typed into a cell (notebooks get committed). Locally it lives
in the repo-root `.env`. On Colab the cloned repo has no `.env`, so the
key comes from **Colab Secrets**: click the key icon in the left
sidebar, add a secret named exactly `OPENAI_API_KEY`, switch on
*Notebook access*. If the secret is missing, the cell asks you to paste
it once (hidden), for this runtime only.

In [ ]:
import utils  # shared helpers from notebooks/utils.py

key_ok = utils.ensure_api_key(IN_COLAB)
assert key_ok, "No OPENAI_API_KEY. Follow the message above, then re-run this cell."

**Why this cell:** every lab talks to a model through
`config/endpoints.py` (Interface Contract #3): one string picks local
Ollama, the hosted API, or the tuned adapter, and the lab code never
changes. Today it is the hosted endpoint. One tiny call proves the key,
the network and the model are all working before the clock starts.

In [ ]:
import json
import time

import fundamentals_utils as fu
from config.endpoints import get_endpoint

llm = get_endpoint("hosted")
print(llm)

ping_result = fu.ping(llm)

## Part A — the 10-minute diagnostic

**Start a 10-minute timer now.** Three probes: two small coding tasks
and four concept questions. Each is a `TODO` with a hint. Work alone,
in order.

- **Stop at 10 minutes wherever you are** and run the score cell. An
  unfinished probe is evidence, not a failure.
- This is not an exam. The result decides how the room spends the next
  45 minutes. A guessed answer helps nobody, least of all you.
- A probe that is left as `...` scores 0 and the notebook carries on.
  Nothing in this section can stop you running the score cell.

What the probes measure: probe 1 — do you know the API has no memory
between calls; probe 2 — can you get a structured answer out of a
model; the quiz — the four concepts the rest of the day assumes.

In [ ]:
# ── TODO 1 (diagnostic probe 1 of 3, about 3 min) ─────────────────
# The user gave their asset tag in an EARLIER turn. Build `messages`
# so the model can answer the follow-up question. One message is
# {"role": ..., "content": ...}; role is "system", "user" or "assistant".
# Hint: the API has no memory of earlier calls. Whatever the model
# should "remember" must be in THIS request's list, in order.
earlier_user = fu.PROBE_1_EARLIER_USER            # "Hi, my laptop is LAP-04412 and it will not boot."
earlier_assistant = fu.PROBE_1_EARLIER_ASSISTANT  # "Sorry to hear that. Does it show any error on screen?"
follow_up = fu.PROBE_1_FOLLOW_UP                  # "No error. Which asset tag did I give you? ..."

messages = ...  # <- replace the ... with your list of messages
# ───────────────────────────────────────────────────────────────────

probe_1_points = fu.score_probe_1(llm, messages)

**Probe 2:** getting a structured answer. The ticket is printed by the
cell; you want it back as a Python dict with exactly three keys.

In [ ]:
# ── TODO 2 (diagnostic probe 2 of 3, about 3 min) ─────────────────
# Get the ticket below back as a Python dict with EXACTLY the keys
# category, urgency and asset_tag.
# Hint: llm.chat_json(prompt, system=...) returns a dict. Tell the
# model the allowed values: category is one of access, hardware,
# software, network, erp, telecom, other; urgency is one of low,
# medium, high, critical; asset_tag is the LAP-xxxxx tag or null.
ticket_text = fu.PROBE_2_TICKET
print(ticket_text)
print()

record = ...  # <- replace the ... with a call that returns a dict
# ───────────────────────────────────────────────────────────────────

probe_2_points = fu.score_probe_2(record)

**Probe 3:** four concepts, one letter each. Leave a question blank if
you would be guessing.

In [ ]:
fu.print_quiz()

# ── TODO 3 (diagnostic probe 3 of 3, about 2 min) ─────────────────
# One letter per question, lowercase.
# Hint: if you are guessing, leave that key out.
quiz_answers = ...  # <- replace with {"q1": "?", "q2": "?", "q3": "?", "q4": "?"}
# ───────────────────────────────────────────────────────────────────

quiz_points = fu.score_quiz(quiz_answers)

**Why this cell:** it turns the three scores into one line the whole
room can read out, and saves it to Drive. The two probes carry half
the points on purpose: nobody reaches READY by guessing the quiz.

**Facilitator:** ask "hands up if your line says READY". READY hands
≥ two thirds of the room → announce **COMPRESSED**. Otherwise announce
**FULL**. Say it once, write it on the board, move on.

In [ ]:
diagnostic = fu.diagnostic_verdict(probe_1_points, probe_2_points, quiz_points)
diagnostic["model"] = llm.model

utils.save_json(CHECKPOINT_DIR, "01_diagnostic", diagnostic)
print("Result saved. Wait for the facilitator's call before going on.")

# ⏸ THE FORK — wait for the facilitator's call

> **FULL path** → carry on into Part B, directly below.
>
> **COMPRESSED path** → scroll down to **▶ PART C STARTS HERE**, click the code cell under that heading, then *Runtime → Run after*. Skip everything marked `[FULL PATH ONLY]`.

## Part B — Fundamentals `[FULL PATH ONLY]` — about 20 minutes

Six short cells: the raw REST call, roles, memory, tokens, temperature,
and one prompt you write yourself. Every code cell in this part starts
with `# [FULL PATH ONLY]`. On the COMPRESSED path, skip to
**▶ PART C STARTS HERE**.

`[FULL PATH ONLY]` **Why this cell:** `llm.chat()` hides exactly one HTTP POST. You call
REST APIs every day, so look at the request and the whole response
once: the reply text is one field inside `choices`, and `usage` tells
you what you paid for. After this cell the helper is a convenience,
not magic. The same request shape works against Ollama on Day 2 and
the tuned adapter — that is why `config/endpoints.py` can switch
between them with one string.

In [ ]:
# [FULL PATH ONLY]
# One POST to /chat/completions. The key goes in the Authorization header and is never printed.
messages = [{"role": "user", "content": "In one sentence, what does an IT service desk do?"}]

body = fu.raw_chat_completion(llm, messages, temperature=0.0, max_tokens=60)

print(json.dumps(body, indent=2)[:1500])
print()
print("reply text  :", fu.reply_text(body))
print("usage       :", body["usage"])
print("finish      :", body["choices"][0]["finish_reason"])

`[FULL PATH ONLY]` **Why this cell:** the **system** message sets who the model is and
the rules it works under; the **user** message is the request. Same
question, two system prompts, two very different answers. This is the
cheapest control you have, and the first thing to reach for before
any fine-tuning or retrieval.

In [ ]:
# [FULL PATH ONLY]
question = "My VPN drops every ten minutes. What should I do?"

terse_desk = "You are an IT service desk bot. Answer in at most two short sentences. No pleasantries."
patient_tutor = "You are a patient tutor explaining networking to a new hire. Use a numbered list of at most four steps."

reply_terse = llm.chat(question, system=terse_desk, temperature=0.0, max_tokens=120)
reply_tutor = llm.chat(question, system=patient_tutor, temperature=0.0, max_tokens=200)

fu.show("system = terse service desk", reply_terse)
fu.show("system = patient tutor", reply_tutor)

`[FULL PATH ONLY]` **Why this cell:** every call is independent. A chat product keeps the
history for you; the API does not. If you want the model to "remember"
the first turn, you send the first turn again, every time, as part of
`messages`. If probe 1 scored 0, this is the cell that explains it.
(Cost follows: a long conversation re-sends its whole history on every
call, and you pay for those tokens each time.)

In [ ]:
# [FULL PATH ONLY]
first_turn = "My laptop is LAP-04412 and it will not boot."
follow_up = "Which asset tag did I give you? Reply with the tag only."

# Attempt 1: the follow-up on its own. The model has never seen the first turn.
reply_alone = llm.chat(follow_up, temperature=0.0, max_tokens=30)

# Attempt 2: the whole conversation, in order, in one request.
history = [
    {"role": "user", "content": first_turn},
    {"role": "assistant", "content": "Understood. Does it show anything on screen?"},
    {"role": "user", "content": follow_up},
]
reply_with_history = llm.chat(messages=history, temperature=0.0, max_tokens=30)

fu.show("follow-up sent alone", reply_alone)
fu.show("follow-up sent with the history", reply_with_history)

`[FULL PATH ONLY]` **Why this cell:** you are billed and limited in **tokens** (roughly
three quarters of a word each), not words or characters. `usage` in
the response counts them. `max_tokens` caps only the **output**, and
the model does not finish its sentence when it hits the cap:
`finish_reason` says `length` instead of `stop`. The context window
(input plus output) is the hard ceiling: a dump of 600 tickets does
not fit in one call, which is why Day 3 is about retrieval.

In [ ]:
# [FULL PATH ONLY]
messages = [{"role": "user", "content": "List five common causes of a laptop failing to boot, one line each."}]

body_short = fu.raw_chat_completion(llm, messages, max_tokens=25)
body_full = fu.raw_chat_completion(llm, messages, max_tokens=300)

for label, body in [("max_tokens=25", body_short), ("max_tokens=300", body_full)]:
    print(f"{label}: finish_reason={body['choices'][0]['finish_reason']}, usage={body['usage']}")
    fu.show(label, fu.reply_text(body))

`[FULL PATH ONLY]` **Why this cell:** `temperature` controls how much randomness goes
into picking each next token. At 0 the same prompt gives (nearly) the
same answer — what you want for extraction, routing and anything you
will test. At 1.2 it varies — fine for brainstorming, poison for a
triage pipeline. Everything else in this notebook runs at 0.

In [ ]:
# [FULL PATH ONLY]
prompt = "Suggest a name for an energy company's internal IT chatbot. One word only."

print("temperature = 0.0")
for attempt in range(3):
    print("  ", llm.chat(prompt, temperature=0.0, max_tokens=8).strip())

print("temperature = 1.2")
for attempt in range(3):
    print("  ", llm.chat(prompt, temperature=1.2, max_tokens=8).strip())

`[FULL PATH ONLY]` **Why this cell:** a prompt is a spec. The parts that matter: the
role, the task, the allowed values, the output format, and what to do
when information is missing. Watch a vague prompt ramble, then write
the spec yourself. This is TODO 4 — the only gap in Part B.

In [ ]:
# [FULL PATH ONLY]
ticket_text = fu.load_sample_tickets(REPO_ROOT, count=1)[0]["text"]
print(ticket_text)
print()

vague_prompt = "Categorise this ticket:\n\n" + ticket_text
reply_vague = llm.chat(vague_prompt, temperature=0.0, max_tokens=150)
fu.show("vague prompt", reply_vague)

# ── TODO 4 (about 5 min) ───────────────────────────────────────────
# Write `good_prompt` so the reply is ONE WORD from this list:
# access, hardware, software, network, erp, telecom, other.
# Hint: state the role, the task, the allowed values, the output format
# ("one word, lowercase, nothing else") and what to do if unsure
# ("other"). Put ticket_text at the end of the prompt.
good_prompt = ...  # <- replace the ... with your prompt string
# ───────────────────────────────────────────────────────────────────
assert good_prompt is not ..., "TODO 4 is not filled in yet"

reply_good = llm.chat(good_prompt, temperature=0.0, max_tokens=10)
fu.show("your prompt", reply_good)

allowed = {"access", "hardware", "software", "network", "erp", "telecom", "other"}
print("one word from the list:", reply_good.strip().lower().strip(".") in allowed)

`[FULL PATH ONLY]` **Why this cell:** milestone. Nothing in Part B was expensive, but
the habit is the point: the results go to `CHECKPOINT_DIR` (Google
Drive on Colab), so a disconnect costs one cell, never the session.
Then continue to Part C.

In [ ]:
# [FULL PATH ONLY]
part_b = {
    "roles_demo": {"terse": reply_terse, "tutor": reply_tutor},
    "memory_demo": {"alone": reply_alone, "with_history": reply_with_history},
    "tokens_demo": {"short_usage": body_short["usage"], "full_usage": body_full["usage"]},
    "prompting_demo": {"vague": reply_vague, "good": reply_good},
}
utils.save_json(CHECKPOINT_DIR, "01_part_b", part_b)
print("Part B done. Continue to Part C.")

# ▶ PART C STARTS HERE — Structured outputs and the agent loop (BOTH PATHS)

**COMPRESSED path: this is your entry point.** Click the code cell
below and use *Runtime → Run after*. Part A (setup and the diagnostic)
must have run in this kernel; nothing from Part B is needed.

About 30 minutes (35 on the compressed path). Three gaps: TODO 5, 6
and 7.

In [ ]:
# Both paths. Guard: Part A must have run in this kernel.
assert "llm" in globals(), "Run Part A first (the setup and diagnostic cells at the top)."

samples = fu.load_sample_tickets(REPO_ROOT, count=5)
print(f"{len(samples)} held-out tickets loaded. The first one:\n")
print(samples[0]["text"])
print("\nexpected record:", json.dumps(samples[0]["expected"]))

**Why this cell:** "return JSON" in a prompt gets you something that
*looks like* JSON. Sometimes it parses; sometimes it arrives wrapped
in a code fence or a polite sentence. A pipeline that calls
`json.loads` on the raw reply works in the demo and fails at 03:00 on
a Tuesday. Look at what comes back, then check it the strict way —
`try_parse_json` is deliberately unforgiving, and `strip_fence` is
the repair small models need most often.

In [ ]:
# Both paths.
ticket_text = samples[0]["text"]
casual_prompt = (
    "Return JSON with the fields category, urgency, asset_tag and requested_action for this ticket:\n\n"
    + ticket_text
)
raw_reply = llm.chat(casual_prompt, temperature=0.0, max_tokens=200)
fu.show("raw reply", raw_reply)

parsed_ok, parsed = fu.try_parse_json(raw_reply)
print("json.loads on the raw reply  :", "OK" if parsed_ok else parsed)

after_fence = fu.strip_fence(raw_reply)
parsed_ok_2, parsed_2 = fu.try_parse_json(after_fence)
print("after stripping a code fence :", "OK" if parsed_ok_2 else parsed_2)

**Why this cell:** valid JSON is not a valid *record*. The locked
ticket schema (BUILD_SPEC section 8B — the one Day 2 fine-tunes
against) says which keys, which enum values, and which pattern an
asset tag must match. `json_mode=True` asks the server to guarantee
JSON **syntax**; the schema validator catches what syntax cannot: an
invented category, a tag with the wrong shape, a missing key. The
system prompt is the Day 2 one, imported, so what you see here is the
untuned baseline that notebook 06 compares against.

In [ ]:
# Both paths.
import jsonschema
from dataset_utils import SYSTEM_PROMPT

schema = fu.load_schema(REPO_ROOT)
validator = jsonschema.Draft202012Validator(schema)
print("schema keys :", list(schema["properties"]))
print("system prompt, first 300 characters:\n", SYSTEM_PROMPT[:300], "...\n")

raw_reply = llm.chat(ticket_text, system=SYSTEM_PROMPT, temperature=0.0, max_tokens=300, json_mode=True)
parsed_ok, record = fu.try_parse_json(raw_reply)
print("JSON syntax :", "OK" if parsed_ok else record)
if parsed_ok:
    errors = fu.schema_errors(record, validator)
    print("schema      :", "valid" if not errors else errors)
    print("record      :", json.dumps(record))
print("expected    :", json.dumps(samples[0]["expected"]))

# What the validator catches that JSON syntax cannot:
broken = dict(samples[0]["expected"], category="printer", asset_tag="LAPTOP-1")
print("\na deliberately broken record:", fu.schema_errors(broken, validator))

**Why this cell:** a model call is a network call to a probabilistic
service. Treat it like any flaky dependency: validate, and on failure
retry **with the error message**, so attempt 2 is better informed than
attempt 1 rather than a re-roll. TODO 5 is the function that builds
that follow-up. It is checked on a made-up failure right here, because
a good model gets most tickets right first time and the gap would
otherwise go untested.

In [ ]:
# Both paths.
# ── TODO 5 (about 8 min) ───────────────────────────────────────────
# Return `messages` extended with two turns: the model's failed reply
# as an "assistant" turn, then a "user" turn that lists the errors and
# asks for a corrected JSON object only.
# Hint: messages.append({"role": ..., "content": ...}) twice, then
# return messages. Put the errors in the text, one per line.
def add_feedback(messages, raw_reply, errors):
    ...  # <- replace the ... with your code
# ───────────────────────────────────────────────────────────────────

# Dry run on a made-up failure, so the gap is checked even when the model is right first time.
demo_messages = add_feedback(
    [{"role": "user", "content": "(a ticket)"}],
    '{"category": "printer"}',
    ["category: 'printer' is not one of ['access', 'hardware', ...]"],
)
assert demo_messages is not None and demo_messages is not ..., "TODO 5 is not filled in yet"
assert len(demo_messages) == 3, "add_feedback must add exactly two turns"
assert demo_messages[1]["role"] == "assistant" and demo_messages[2]["role"] == "user", "assistant turn first, then user"
print("add_feedback OK. The follow-up turn reads:\n")
print(demo_messages[2]["content"])

**Why this cell:** the ask-validate-retry loop, in full. Read it top
to bottom once: it is the shape of every production call you will
write this week. Three attempts, then a loud failure rather than a
quiet bad record.

In [ ]:
# Both paths.
def extract_record(ticket_text, max_attempts=3):
    # Ask, validate, and on failure ask again with the errors. Returns (record, attempts).
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ticket_text},
    ]
    for attempt in range(1, max_attempts + 1):
        raw = llm.chat(messages=messages, temperature=0.0, max_tokens=300, json_mode=True)
        parsed_ok, record = fu.try_parse_json(raw)
        if parsed_ok:
            errors = fu.schema_errors(record, validator)
        else:
            errors = [record]  # the parse error message
        if not errors:
            return record, attempt
        print(f"  attempt {attempt} failed: {errors}")
        messages = add_feedback(messages, raw, errors)
    raise ValueError(f"no valid record after {max_attempts} attempts")


record, attempts = extract_record(samples[0]["text"])
print(f"valid record after {attempts} attempt(s):", json.dumps(record))

**Why this cell:** one ticket proves nothing. Run all five held-out
tickets and compare field by field with the labelled records. Look at
the *shape* of the misses: the schema is fine; the **content** is
where a general model disagrees with house conventions (routing
queues, urgency). Day 2 fine-tunes exactly that gap, and notebook 06
scores the same 20 held-out tickets. Milestone: the rows go to Drive,
and re-running this cell loads them instead of paying for the calls
again.

In [ ]:
# Both paths. Milestone: load if the checkpoint exists, compute if not.
extraction_rows = utils.load_json(CHECKPOINT_DIR, "01_extractions", default=None)

if extraction_rows is None:
    extraction_rows = []
    for sample in samples:
        record, attempts = extract_record(sample["text"])
        verdicts = fu.compare_records(record, sample["expected"])
        extraction_rows.append({"ticket_id": sample["ticket_id"], "attempts": attempts,
                                "record": record, "verdicts": verdicts})
    utils.save_json(CHECKPOINT_DIR, "01_extractions", extraction_rows)
    came_from = "fresh run"
else:
    came_from = "checkpoint"

fu.print_comparison_table(extraction_rows)
print(f"\n{len(extraction_rows)} tickets (from {came_from}). ok = matches the labelled record; "
      "act = word overlap >= 0.5 with the labelled requested_action.")

**Why this cell:** an agent is a loop **you** write. The model decides
what to do next; your code does it; the result goes back as a message;
repeat until the model says it is done. The model never executes
anything. Here the "tools" are two plain Python functions over
synthetic data, and the model asks for one by replying with a small
JSON action. The hosted API's native tool calling does the same thing
through a dedicated `tools` field and a `tool` role; Day 4 uses that.
The mechanics are identical, and here you can see every moving part.

In [ ]:
# Both paths. Two tools: plain functions with a name, arguments and a return value.
def get_ticket(ticket_id):
    return fu.get_ticket(REPO_ROOT, ticket_id)


def get_asset(asset_tag):
    return fu.get_asset(asset_tag)


TOOLS = {"get_ticket": get_ticket, "get_asset": get_asset}

print(json.dumps(get_ticket("INC-004549"), indent=2)[:700])
print(json.dumps(get_asset("LAP-04391"), indent=2))
print()
print(fu.AGENT_SYSTEM_PROMPT)

**Why this cell:** the whole agent is this one cell: ask, parse the
action, run the tool, append the observation, ask again. A step cap
stops a confused model from looping forever, and every step is
printed, because an agent you cannot watch is an agent you cannot
debug. TODO 6 is the dispatch: turning the model's request into a
function call.

In [ ]:
# Both paths.
QUESTION = ("Ticket INC-004549 reports a laptop problem. Is that laptop still under warranty "
            "on the date the asset register gives as today, and should we repair or replace it? "
            "Use the tools; do not guess.")
MAX_STEPS = 6

messages = [
    {"role": "system", "content": fu.AGENT_SYSTEM_PROMPT},
    {"role": "user", "content": QUESTION},
]
trace = []
final_answer = None

for step in range(1, MAX_STEPS + 1):
    raw = llm.chat(messages=messages, temperature=0.0, max_tokens=300, json_mode=True)
    action = fu.parse_action(raw)
    messages.append({"role": "assistant", "content": raw})

    if "final" in action:
        final_answer = action["final"]
        print(f"step {step}: FINAL -> {final_answer}")
        break

    # ── TODO 6 (about 8 min) ───────────────────────────────────────
    # Run the tool the model asked for. `action` looks like
    # {"tool": "get_asset", "args": {"asset_tag": "LAP-04391"}}.
    # Look the name up in TOOLS, call it with the args, and put the
    # result in `observation`. If the tool name is unknown, make the
    # observation an error dict instead of crashing: the model can
    # recover from an error message, not from an exception.
    # Hint: TOOLS[name](**args)
    observation = ...  # <- replace the ...
    # ───────────────────────────────────────────────────────────────
    assert observation is not ..., "TODO 6 is not filled in yet"

    print(f"step {step}: {action.get('tool')}({action.get('args')}) -> {json.dumps(observation)[:120]}")
    trace.append({"step": step, "action": action, "observation": observation})
    messages.append({"role": "user", "content": "Tool result: " + json.dumps(observation)})

if final_answer is None:
    print(f"no final answer after {MAX_STEPS} steps: the model is looping. Read the trace.")

**Why this cell:** the trace is the artefact worth keeping (milestone),
and two questions make sure the loop was read, not just run. Day 5's
approval interrupt and Day 4's control lab both start from your answer
to the first one.

In [ ]:
# Both paths. Milestone: the trace is what you would show an auditor.
agent_result = {"question": QUESTION, "tool_calls": len(trace),
                "finished": final_answer is not None, "final_answer": final_answer, "trace": trace}
utils.save_json(CHECKPOINT_DIR, "01_agent", agent_result)

# ── TODO 7 (about 5 min) ───────────────────────────────────────────
# Two sentences, as strings. (1) Which single line of the loop above
# would you change so the agent asks a human before calling a tool that
# WRITES (say, close_ticket)? (2) Why does the loop need MAX_STEPS?
# Hint: look at where `observation` is produced, and at what happens
# when the model never replies with "final".
reflection = ...  # <- {"approval_gate": "...", "step_cap": "..."}
# ───────────────────────────────────────────────────────────────────
assert reflection is not ..., "TODO 7 is not filled in yet"

utils.save_json(CHECKPOINT_DIR, "01_reflection", reflection)
print("Part C done. Run the final cell.")

**Why this cell:** the last cell prints the result the header
promised, from the checkpoints, so "done" is checkable at a glance
on either path — and after a reconnect.

In [ ]:
# The header promised: 01 FUNDAMENTALS OK + diagnostic verdict + schema-valid count + agent steps.
diagnostic = utils.load_json(CHECKPOINT_DIR, "01_diagnostic")
extraction_rows = utils.load_json(CHECKPOINT_DIR, "01_extractions")
agent_result = utils.load_json(CHECKPOINT_DIR, "01_agent")

schema_valid = sum(1 for row in extraction_rows if fu.schema_errors(row["record"], validator) == [])
whole_record = sum(1 for row in extraction_rows if all(row["verdicts"][f] for f in fu.EXACT_FIELDS))
agent_steps = agent_result["tool_calls"] + (1 if agent_result["finished"] else 0)

all_good = agent_result["finished"] and schema_valid == len(extraction_rows)
fu.banner("01 FUNDAMENTALS OK" if all_good else "01 FUNDAMENTALS: CHECK THE LINES BELOW")
print(f"environment   : {'Colab' if IN_COLAB else 'local'}")
print(f"diagnostic    : {diagnostic['verdict']} ({diagnostic['total']} / {fu.DIAGNOSTIC_MAX})")
print(f"schema-valid  : {schema_valid} / {len(extraction_rows)} held-out tickets")
print(f"whole record  : {whole_record} / {len(extraction_rows)} (six exact fields; the untuned baseline Day 2 improves on)")
print(f"agent         : {'final answer in ' + str(agent_steps) + ' steps' if agent_result['finished'] else 'NO final answer'}")